# Polars — User Journey Benchmark

Reads the LaminDB collection as a lazy Polars LazyFrame via `collection.open(engine="polars")`.
Queries use native Polars expressions with automatic predicate and projection pushdown.
No ingestion step — data stays in LaminDB.

`QUERY_SOURCE = "store"` re-reads S3 on each query; `"memory"` materialises once.
Run top to bottom.

In [1]:
import polars as pl
import pandas as pd
import lamindb as ln

from bench_utils import Bench, stats_pandas, recurrent_pandas, filter_params, make_append_batch

# --- config ---
QUERY_SOURCE   = "store"
COLLECTION_UID = "hVu9puwdRGskm1I60000"
SCHEMA_NAME    = "1000 Genomes CNV VCF"

ln.track("2Wdo02w0MDgH", project="Lakehouse benchmarks v1")
b = Bench("polars")
collection = ln.Collection.get(COLLECTION_UID, is_latest=False)

# open as a Polars LazyFrame — nothing is downloaded until .collect() is called
_ctx = collection.open(engine="polars")
lazy_df = _ctx.__enter__()             # context manager; holds the S3-backed LazyFrame

→ connected lamindb: laminlabs/lakehouse-benchmarks


→ loaded Transform('2Wdo02w0MDgH0006', key='polars_pipeline.ipynb'), started new Run('hNyU6jKLLKVdpqAF') at 2026-07-06 23:13:03 UTC


→ notebook imports: bench_utils lamindb-core==2.7.0 pandas==2.3.3 polars==1.42.0


In [2]:
# --- read: footers-only in store mode, full materialise in memory mode ---
def read_from_lamin(query_source: str):
    return lazy_df.collect() if query_source == "memory" else None

with b("read_from_lamin"):
    df_mem = read_from_lamin(QUERY_SOURCE)

rows_initial = len(df_mem) if df_mem is not None else lazy_df.select(pl.len()).collect().item()
assert rows_initial > 0
print(f"Total rows: {rows_initial:,}")

b.timings["ingest"] = 0.0              # no ingest — data stays in LaminDB

def query_df() -> pd.DataFrame:
    # bench_utils functions expect pandas; convert at the boundary
    df = df_mem if df_mem is not None else lazy_df.collect()
    return df.to_pandas()

  read_from_lamin: 0.000s


Total rows: 88,332,015


In [3]:
# --- filtered query: Polars lazy filter with predicate + projection pushdown ---
# projection pushdown: read only 2 columns to derive filter bounds
fp_df = (df_mem.select(["chrom", "pos"]) if df_mem is not None
         else lazy_df.select(["chrom", "pos"]).collect()).to_pandas()
CHROM, LO, HI = filter_params(fp_df)
print(f"filter: chrom={CHROM!r} pos in [{LO:,}, {HI:,}]")

def filtered_query(chrom: str, lo: int, hi: int) -> pl.DataFrame:
    expr = (pl.col("chrom") == chrom) & (pl.col("pos") >= lo) & (pl.col("pos") <= hi)
    if df_mem is not None:
        return df_mem.filter(expr)
    return lazy_df.filter(expr).collect()

with b("filtered_query"):
    filtered = filtered_query(CHROM, LO, HI)
assert len(filtered) > 0
print(f"Variants in {CHROM}:{LO}-{HI}: {len(filtered):,}")

filter: chrom='2' pos in [22,622,434, 220,406,165]


  filtered_query: 2.412s
Variants in 2:22622434-220406165: 5,665,280


In [4]:
# --- query 2: per-sample statistics ---
def query_stats(query_source: str):
    return stats_pandas(query_df())

with b("query_stats"):
    stats = query_stats(QUERY_SOURCE)
assert len(stats) > 0
stats.head()

  query_stats: 77.081s


,Chrom,Total_Variants,Deletions,Insertions,Mean_AF,Mean_EUR_AF
0,1,6468094,0,0,0.036494,0.034716
1,10,3992219,0,0,0.037922,0.036399
2,11,4045628,0,0,0.038082,0.037075
3,12,3868428,0,0,0.037274,0.035666
4,13,2857916,0,0,0.040664,0.039962


In [5]:
# --- query 3: recurrent CNV regions ---
def query_recurrent(query_source: str):
    return recurrent_pandas(query_df())

with b("query_recurrent"):
    recurrent = query_recurrent(QUERY_SOURCE)
assert len(recurrent) > 0
print(f"Identified {len(recurrent)} recurrent regions.")

  query_recurrent: 111.298s
Identified 2911 recurrent regions.


In [6]:
# --- append: schema-validated artifact + new collection version ---
# make_append_batch needs Arrow; .to_arrow() is a zero-copy cast from Polars
batch_arrow = lazy_df.filter(pl.col("chrom") == CHROM).limit(2000).collect().to_arrow()
batch = make_append_batch(batch_arrow)
# add run uid column so content hash is unique per run → avoids dedup lineage tangle
batch_df = pl.from_arrow(batch).with_columns(
    pl.lit(ln.context.run.uid).alias("benchmark_run")
).to_pandas()                           # LaminDB from_dataframe expects pandas
APPEND_ROWS = len(batch_df)

new_art = ln.Artifact.from_dataframe(
    batch_df,
    key=f"lakehouse-benchmarks/append_batch_{ln.context.run.uid}.parquet",
    description="benchmark append batch (new sample)",
).save()

original = ln.Collection.get(COLLECTION_UID)
original_arts = [a for a in original.artifacts.all() if "append_batch" not in (a.key or "")]


def append_new_sample() -> ln.Collection:
    return ln.Collection(
        [*original_arts, new_art],
        key=original.key, description=original.description, revises=original,
    ).save()

with b("append"):
    new_collection = append_new_sample()

rows_after = sum(a.open().count_rows() for a in new_collection.artifacts.all())
assert rows_after == rows_initial + APPEND_ROWS, (rows_after, rows_initial, APPEND_ROWS)
print(f"Rows after append: {rows_after:,} (+{APPEND_ROWS})")

... uploading XVTFdMaCq7TUXpNs0000.parquet:  0.0%

... uploading XVTFdMaCq7TUXpNs0000.parquet: 100.0%


  append: 4.223s


Rows after append: 88,334,015 (+2000)


In [7]:
# --- schema change: update the registered schema (validates all future artifacts) ---
schema = ln.Schema.get(name=SCHEMA_NAME)

def evolve_schema(feature_name: str):
    feat = ln.Feature(name=feature_name, dtype=bool).save()
    schema.add_optional_features([feat])

with b("schema_change"):
    evolve_schema("QC_PASS")
print("Schema now includes QC_PASS")

→ returning feature with same name: 'QC_PASS'


  schema_change: 3.500s
Schema now includes QC_PASS


In [8]:
# --- governance demos ---
# 1. Closed schema rejects bad data at SAVE time.
members = list(ln.Schema.get(name=SCHEMA_NAME).members.all())
strict = ln.Schema(features=members, minimal_set=True, maximal_set=True).save()

bad = pd.DataFrame({"wrong_column": [1, 2, 3]})
try:
    ln.Artifact.from_dataframe(
        bad, key="lakehouse-benchmarks/should_fail.parquet",
        description="invalid artifact (governance demo)", schema=strict,
    ).save()
    print("governance: WARNING — invalid artifact was NOT rejected")
except Exception as e:
    print(f"governance: schema validation rejected bad data ({type(e).__name__})")

→ returning schema with same hash: Schema(uid='kgenuYjuaxXSD6rd', is_type=False, name=None, description=None, n_members=1, coerce=None, flexible=False, itype='Feature', otype=None, hash='2IDUCp8Mb2F2Xj3nWfaXpw', minimal_set=True, ordered_set=False, maximal_set=True, branch_id=1, created_on_id=1, space_id=1, created_by_id=2, run_id=148, type_id=None, created_at=2026-06-24 08:39:37 UTC, is_locked=False)


→ loading artifact into memory for validation


governance: schema validation rejected bad data (ValidationError)


In [9]:
_ctx.__exit__(None, None, None)        # close the Polars context manager cleanly
b.record(query_source=QUERY_SOURCE)
ln.finish()

→ creating new artifact version for key 'benchmark_results/polars.parquet' in storage 's3://lamin-eu-central-1/MBiQHz7l46Jk'


... uploading fw91ufbHObjVGed00008.parquet:  0.0%

... uploading fw91ufbHObjVGed00008.parquet: 100.0%


• replacing the existing cache path /home/sagemaker-user/.cache/lamindb/lamin-eu-central-1/MBiQHz7l46Jk/benchmark_results/polars.parquet



recorded 7 steps -> benchmark_results/polars.parquet


→ to save the notebook html, run: lamin save /home/sagemaker-user/lakehouse-benchmarks/polars_pipeline.ipynb


→ finished Run('hNyU6jKLLKVdpqAF') after 5m at 2026-07-06 23:18:15 UTC


→ go to: https://lamin.ai/laminlabs/lakehouse-benchmarks/run/hNyU6jKLLKVdpqAF


→ to update your notebook from the CLI, run: lamin save /home/sagemaker-user/lakehouse-benchmarks/polars_pipeline.ipynb
